In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

In [2]:
holidays = ["2023-01-02", "2023-01-16", "2023-02-13", "2023-02-20", "2023-05-29", "2023-06-19", "2023-07-04", "2023-09-04", "2023-10-09", "2023-11-10", "2023-11-23", "2023-12-25", "2024-01-01", "2024-01-15", "2024-02-12", "2024-02-19", "2024-05-27", "2024-06-19", "2024-07-04", "2024-09-02", "2024-10-14", "2024-11-11", "2024-11-28", "2024-12-25", "2025-01-01", "2025-01-20", "2025-02-12", "2025-02-17", "2025-05-26", "2025-06-19", "2025-07-04", "2025-09-01", "2025-10-13", "2025-11-11", "2025-11-27", "2025-12-25"]

In [3]:
spark = SparkSession.builder.config("spark.driver.memory", "6g").config("spark.driver.maxResultSize", "2g").master("local[*]").appName("traffic_speed_anomalies").getOrCreate()
base_path = "/home/jovyan/work"

In [4]:
df = spark.read.parquet(f"{base_path}/data/traffic_speeds/cleaned_traffic_speeds/")
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)



In [9]:
df_dim = df \
    .select("ID", "LINK_POINTS", "BOROUGH") \
    .distinct()

df_dim.printSchema()

root
 |-- ID: string (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)



In [10]:
print(df_dim.count())

120


In [11]:
df_dim.write.parquet(f"{base_path}/data/traffic_speeds/traffic_speeds_anomalies/dim/")

In [5]:
df_fact = df \
    .select("ID", "DATE", "SPEED", "TRAVEL_TIME")

df_fact.printSchema()

root
 |-- ID: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)



In [17]:
print(df_fact.count())

28791002


In [6]:
df_fact = df_fact \
    .filter(~col("DATE").isin(holidays)) \
    .withColumn("TIME", date_format(to_timestamp(floor(unix_timestamp(to_timestamp("DATE", "HH:mm:ss")) / 300) * 300).cast("timestamp"), "HH:mm:ss")) \
    .withColumn("day_type", when(weekday(col("DATE")) == 5, "sat").when(weekday(col("DATE")) == 6, "sun").otherwise("weekday"))

df_fact.printSchema()

root
 |-- ID: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- day_type: string (nullable = false)



In [19]:
print(df_fact.count())

28791002


In [7]:
n = 50

counts = df_fact.groupBy("ID", "day_type", "TIME").count()

df_fact = df_fact.join(counts, on=["ID", "day_type", "TIME"], how="left")

df_fact = df_fact \
    .filter(col("count") > n) \
    .drop("count")

df_fact.printSchema()

root
 |-- ID: string (nullable = true)
 |-- day_type: string (nullable = false)
 |-- TIME: string (nullable = true)
 |-- DATE: timestamp (nullable = true)
 |-- SPEED: string (nullable = true)
 |-- TRAVEL_TIME: string (nullable = true)



In [8]:
print(df_fact.count())

28644238


In [8]:
df_fact.write.parquet(f"{base_path}/data/traffic_speeds/traffic_speeds_anomalies/fact/")

In [9]:
df_fact = spark.read.parquet(f"{base_path}/data/traffic_speeds/traffic_speeds_anomalies/fact/")

In [20]:
df_fact_median = df_fact \
    .groupBy("ID", "day_type", "TIME") \
    .agg(percentile_approx("SPEED", 0.5).alias("MEDIAN_SPEED"))

df_fact_median.printSchema()

root
 |-- ID: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- MEDIAN_SPEED: double (nullable = true)



In [21]:
df_fact_median = df_fact \
    .join(df_fact_median, on=["ID", "day_type", "TIME"], how="inner")

df_fact_median = df_fact_median \
    .withColumn("abs_deviation", abs(col("SPEED") - col("MEDIAN_SPEED"))) \
    .groupBy("ID", "day_type", "TIME", "MEDIAN_SPEED") \
    .agg(percentile_approx("abs_deviation", 0.5).alias("MAD"))

counts = df_fact_median.groupBy("ID", "day_type").count()

df_fact_median = df_fact_median \
    .join(counts, on=["ID", "day_type"], how="left")

In [22]:
n = 288

df_fact_median = df_fact_median.filter(col("count") >= n).select("ID", "day_type", "TIME", "MEDIAN_SPEED", "MAD")

In [23]:
df_fact_median.printSchema()

root
 |-- ID: string (nullable = true)
 |-- day_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- MEDIAN_SPEED: double (nullable = true)
 |-- MAD: double (nullable = true)



In [14]:
df_fact_median.show(5)

+---+--------+--------+------------+------------------+
| ID|day_type|    TIME|MEDIAN_SPEED|               MAD|
+---+--------+--------+------------+------------------+
|172|     sat|21:30:00|       15.53|              4.35|
|172|     sat|21:00:00|       14.91|3.1099999999999994|
|172|     sat|19:50:00|       13.67|              2.49|
|172|     sat|19:35:00|       13.67|3.1099999999999994|
|172|     sat|18:25:00|       12.42|3.0999999999999996|
+---+--------+--------+------------+------------------+
only showing top 5 rows



In [24]:
df_fact_median = df_fact_median \
    .withColumn("MIN_SPEED_THRESHOLD", greatest(lit(0.0), col("MEDIAN_SPEED") - ((3.5 / 0.6745) * col("MAD"))))

In [25]:
df_fact_median.show(5)

+---+--------+--------+------------+------------------+-------------------+
| ID|day_type|    TIME|MEDIAN_SPEED|               MAD|MIN_SPEED_THRESHOLD|
+---+--------+--------+------------+------------------+-------------------+
|172|     sat|21:30:00|       15.53|              4.35|                0.0|
|172|     sat|21:00:00|       14.91|3.1099999999999994|                0.0|
|172|     sat|19:50:00|       13.67|              2.49| 0.7493180133432169|
|172|     sat|19:35:00|       13.67|3.1099999999999994|                0.0|
|172|     sat|18:25:00|       12.42|3.0999999999999996|                0.0|
+---+--------+--------+------------+------------------+-------------------+
only showing top 5 rows



In [28]:
df_fact_median.write.mode("overwrite").partitionBy('ID').parquet(f"{base_path}/data/traffic_speeds/traffic_speeds_anomalies/fact/")

In [7]:
df_speed_median = spark.read.parquet('traffic_speeds_2023_2025_median')

In [8]:
df_anomalies = df_speed \
    .join(df_speed_median, on=["ID", "day_type", "TIME"], how="inner")

df_anomalies.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- day_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- SPEED: float (nullable = true)
 |-- TRAVEL_TIME: integer (nullable = true)
 |-- DATE: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- count: long (nullable = true)
 |-- MEDIAN_SPEED: float (nullable = true)
 |-- MAD: float (nullable = true)



In [11]:
df_anomalies = df_anomalies \
    .withColumn("z_score", (0.6745 * (df_anomalies.SPEED - df_anomalies.MEDIAN_SPEED)) / df_anomalies.MAD)

df_anomalies = df_anomalies \
    .filter(df_anomalies.z_score < (-3.5)) \
    .withColumn("TIMESTAMP", to_timestamp(concat_ws(" ", df_anomalies.DATE, df_anomalies.TIME), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("time_minutes", (unix_timestamp("TIMESTAMP") / 60).cast("long"))

df_anomalies.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- day_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- SPEED: float (nullable = true)
 |-- TRAVEL_TIME: integer (nullable = true)
 |-- DATE: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- count: long (nullable = true)
 |-- MEDIAN_SPEED: float (nullable = true)
 |-- MAD: float (nullable = true)
 |-- z_score: double (nullable = true)
 |-- TIMESTAMP: timestamp (nullable = true)
 |-- time_minutes: long (nullable = true)



In [12]:
w = Window.partitionBy("ID").orderBy("time_minutes")

df_anomalies = df_anomalies \
    .withColumn("row_number", row_number().over(w))

df_anomalies = df_anomalies \
    .withColumn("island_id", df_anomalies["time_minutes"] - df_anomalies["row_number"] * 5)

In [13]:
df_anomalies.show(5)

+---+--------+--------+-----+-----------+----------+----+-----+-----+------------+---------+-------------------+-------------------+------------+----------+---------+
| ID|day_type|    TIME|SPEED|TRAVEL_TIME|      DATE|year|month|count|MEDIAN_SPEED|      MAD|            z_score|          TIMESTAMP|time_minutes|row_number|island_id|
+---+--------+--------+-----+-----------+----------+----+-----+-----+------------+---------+-------------------+-------------------+------------+----------+---------+
|126|     sun|01:05:00|42.25|        177|2023-01-01|2023|    1|  148|       53.43|1.8600006|-4.0542514684528745|2023-01-01 01:05:00|    27875585|         1| 27875580|
|126|     sun|01:10:00|41.63|        180|2023-01-01|2023|    1|  149|       53.43|     1.25| -6.367279588317871|2023-01-01 01:10:00|    27875590|         2| 27875580|
|126|     sun|01:15:00|42.25|        175|2023-01-01|2023|    1|  147|       53.43|1.8699989| -4.032574605933387|2023-01-01 01:15:00|    27875595|         3| 27875580

In [14]:
n = 3

anomaly_islands = df_anomalies \
    .groupBy("ID", "island_id") \
    .agg(count("*").alias("size"))

df_anomalies = df_anomalies \
    .join(anomaly_islands.filter(anomaly_islands.size >= n) \
        .select("ID", "island_id"), on=["ID", "island_id"], how="leftsemi")

In [15]:
df_anomalies = df_anomalies \
    .groupBy("ID", "island_id") \
    .agg(min("TIMESTAMP").alias("anomaly_start"), max("TIMESTAMP").alias("anomaly_end"))

In [18]:
print(df_anomalies.count())

129843


In [6]:
spark.stop()